# 03k — Análisis de errores por subgrupo (churn_30d)

Sobre test set + modelo CatBoost tuneado:
- Métricas por country, antigüedad, actividad, gasto
- Top 20 false positives + false negatives

In [1]:
# [SETUP]
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from pathlib import Path
from datetime import datetime

TARGET = 'churn_30d'
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03k_error_analysis'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = DATA_QC / 'master_table_cutoff_v3_aggressive.parquet'
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'
MODEL_PATH = DATA_MODELS / f'catboost_tuned_{TARGET}_v3.cbm'

In [2]:
# [EXEC] Cargar datos + modelo
master = pd.read_parquet(MASTER_PATH).reset_index(drop=True)
splits_df = pd.read_parquet(SPLITS_PATH).reset_index(drop=True)
if 'user_id' in splits_df.columns:
    splits_df = splits_df.set_index('user_id').reindex(master['user_id'].values).reset_index()

cat_cols = master.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'user_id']
for c in cat_cols:
    master[c] = master[c].astype(str).fillna('missing').replace('nan', 'missing')

test_mask = splits_df['split'].values == 'test'
X_test = master.drop(columns=['churn_14d', 'churn_30d', 'user_id'])[test_mask].reset_index(drop=True)
y_test = master[TARGET].astype(int)[test_mask].reset_index(drop=True)
user_id_test = master.loc[test_mask, 'user_id'].reset_index(drop=True)
test_df = master[test_mask].reset_index(drop=True).copy()

model = CatBoostClassifier()
model.load_model(str(MODEL_PATH))
test_pool = Pool(X_test, y_test, cat_features=cat_cols)
y_pred_proba = model.predict_proba(test_pool)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred
test_df['y_pred_proba'] = y_pred_proba
test_df['error_type'] = 'TN'
test_df.loc[(test_df['y_true']==1) & (test_df['y_pred']==1), 'error_type'] = 'TP'
test_df.loc[(test_df['y_true']==0) & (test_df['y_pred']==1), 'error_type'] = 'FP'
test_df.loc[(test_df['y_true']==1) & (test_df['y_pred']==0), 'error_type'] = 'FN'

print(f'Test set: {len(test_df):,} usuarios')
print(test_df['error_type'].value_counts())

Test set: 3,780 usuarios
error_type
TN    1735
TP    1054
FN     739
FP     252
Name: count, dtype: int64


In [3]:
# [EXEC] Métricas por subgrupo
def metrics_by_group(df, group_col, group_name):
    rows = []
    for val, sub in df.groupby(group_col):
        if len(sub) < 30: continue  # ignorar grupos pequeños
        try:
            auc = roc_auc_score(sub['y_true'], sub['y_pred_proba']) if sub['y_true'].nunique() > 1 else float('nan')
        except Exception:
            auc = float('nan')
        f1 = f1_score(sub['y_true'], sub['y_pred'], zero_division=0)
        rows.append({
            'group_dim': group_name,
            'group_value': str(val),
            'n': len(sub),
            'churn_rate': float(sub['y_true'].mean()),
            'auc_roc': auc,
            'f1': f1,
            'fpr': ((sub['y_true']==0) & (sub['y_pred']==1)).sum() / max(1, (sub['y_true']==0).sum()),
            'fnr': ((sub['y_true']==1) & (sub['y_pred']==0)).sum() / max(1, (sub['y_true']==1).sum()),
        })
    return pd.DataFrame(rows).sort_values('n', ascending=False)

all_subgroup_metrics = []

# 1. Por country (top 5 vs resto)
if 'country' in test_df.columns:
    top5_countries = test_df['country'].value_counts().head(5).index.tolist()
    test_df['country_grouped'] = test_df['country'].where(test_df['country'].isin(top5_countries), 'OTHER')
    all_subgroup_metrics.append(metrics_by_group(test_df, 'country_grouped', 'country'))

# 2. Por quartiles de antigüedad de cuenta
if 'user_created_at_days_ago' in test_df.columns:
    test_df['account_age_q'] = pd.qcut(test_df['user_created_at_days_ago'].fillna(-1), q=4, duplicates='drop', labels=['Q1_new', 'Q2', 'Q3', 'Q4_old'])
    all_subgroup_metrics.append(metrics_by_group(test_df, 'account_age_q', 'account_age'))

# 3. Por quartiles de inactividad device
if 'device_days_since_last_active' in test_df.columns:
    test_df['device_inactivity_q'] = pd.qcut(test_df['device_days_since_last_active'].clip(upper=9000), q=4, duplicates='drop', labels=['Q1_active', 'Q2', 'Q3', 'Q4_inactive'])
    all_subgroup_metrics.append(metrics_by_group(test_df, 'device_inactivity_q', 'device_inactivity'))

# 4. Por payer / no-payer
if 'iap_is_payer' in test_df.columns:
    test_df['payer_flag'] = test_df['iap_is_payer'].fillna(0).astype(int).map({0: 'non_payer', 1: 'payer'})
    all_subgroup_metrics.append(metrics_by_group(test_df, 'payer_flag', 'payer'))

subgroup_df = pd.concat(all_subgroup_metrics, ignore_index=True)
subgroup_df.to_csv(REPORT_DIR / 'metrics_by_subgroup.csv', index=False)
print(subgroup_df.round(4).to_string(index=False))

        group_dim   group_value    n  churn_rate  auc_roc     f1    fpr    fnr
          country         OTHER 2295      0.5015   0.8111 0.7185 0.1538 0.3536
          country        Russia  650      0.3631   0.6819 0.4337 0.0580 0.6949
          country United States  404      0.3936   0.7740 0.6111 0.0653 0.5157
          country        Brazil  185      0.5676   0.7720 0.7104 0.1625 0.3810
          country       Ukraine  126      0.4762   0.8366 0.6598 0.0758 0.4667
          country          Iran  120      0.6833   0.7892 0.7805 0.4737 0.2195
      account_age        Q1_new  953      0.7901   0.8475 0.8962 0.7000 0.0372
      account_age            Q2  945      0.4635   0.7219 0.5647 0.1538 0.5365
      account_age        Q4_old  942      0.2824   0.6421 0.3158 0.0089 0.8083
      account_age            Q3  940      0.3574   0.6611 0.3417 0.0464 0.7768
device_inactivity     Q1_active  950      0.8189   0.8474 0.9135 0.7733 0.0154
device_inactivity   Q4_inactive  945      0.2804   0

In [4]:
# [EXEC] Top 20 FP y FN — guardar para inspección
fp_top = test_df[test_df['error_type']=='FP'].nlargest(20, 'y_pred_proba')
fn_top = test_df[test_df['error_type']=='FN'].nsmallest(20, 'y_pred_proba')
cols_keep = ['user_id', 'y_true', 'y_pred', 'y_pred_proba']
for c in ['country', 'language', 'user_created_at_days_ago', 'device_days_since_last_active',
         'reward_last_claim_days_ago', 'char_last_updated_days_ago']:
    if c in test_df.columns:
        cols_keep.append(c)
fp_top[cols_keep].to_csv(REPORT_DIR / 'top20_false_positives.csv', index=False)
fn_top[cols_keep].to_csv(REPORT_DIR / 'top20_false_negatives.csv', index=False)
print(f'FP top 20 (predijo churn pero NO churn):\n{fp_top[cols_keep].head(10).to_string(index=False)}')
print(f'\nFN top 20 (predijo activo pero SÍ churn):\n{fn_top[cols_keep].head(10).to_string(index=False)}')

FP top 20 (predijo churn pero NO churn):
                 user_id  y_true  y_pred  y_pred_proba     country  language  user_created_at_days_ago  device_days_since_last_active  reward_last_claim_days_ago  char_last_updated_days_ago
692af46b2b04740f6c092783       0       1      0.974512 Philippines       0.0                         5                              6                           5                           4
691f1fed4ec8bc2b491f3ee1       0       1      0.952698      Brazil       1.0                        14                             15                           1                           1
6930b1d8c8d8b450a30fdabf       0       1      0.941219       Egypt       0.0                         1                              1                           1                           1
69121a50ab71293403213844       0       1      0.935056  Kazakhstan       7.0                        24                             25                           4                           4
692f43835

In [5]:
# [REPORT] error_analysis_summary.md
cm_global = confusion_matrix(test_df['y_true'], test_df['y_pred'])
tn, fp, fn, tp = cm_global.ravel()
auc_global = roc_auc_score(test_df['y_true'], test_df['y_pred_proba'])

lines = [
    f'# Análisis de errores — CatBoost tuneado ({TARGET})',
    '',
    f'**Fecha:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'**Modelo:** catboost_tuned_{TARGET}_v3.cbm',
    f'**Test set:** {len(test_df):,} usuarios',
    '',
    '## Confusion matrix global',
    '',
    '|  | pred=0 | pred=1 |',
    '|---|---:|---:|',
    f'| **true=0** | TN={tn} | FP={fp} |',
    f'| **true=1** | FN={fn} | TP={tp} |',
    '',
    f'- **AUC global**: {auc_global:.4f}',
    f'- **FPR**: {fp/(fp+tn):.4f} ({fp}/{fp+tn})',
    f'- **FNR**: {fn/(fn+tp):.4f} ({fn}/{fn+tp})',
    f'- **Precision**: {tp/(tp+fp):.4f}',
    f'- **Recall**: {tp/(tp+fn):.4f}',
    '',
    '## Métricas por subgrupo',
    '',
    '(filtrado: solo grupos con n≥30)',
    '',
    '| group_dim | group_value | n | churn_rate | AUC | F1 | FPR | FNR |',
    '|---|---|---:|---:|---:|---:|---:|---:|',
]
for _, row in subgroup_df.iterrows():
    auc_str = f'{row["auc_roc"]:.4f}' if pd.notna(row['auc_roc']) else 'n/a'
    lines.append(
        f'| {row["group_dim"]} | {row["group_value"]} | {row["n"]} | {row["churn_rate"]:.4f} | '
        f'{auc_str} | {row["f1"]:.4f} | {row["fpr"]:.4f} | {row["fnr"]:.4f} |'
    )

lines += [
    '',
    '## Patrones detectados',
    '',
    '- **AUC más alto** (mejor performance del modelo) en grupos con: ',
    '  ' + ', '.join(subgroup_df.dropna(subset=['auc_roc']).nlargest(3, 'auc_roc')['group_value'].tolist()),
    '- **AUC más bajo** (oportunidad de mejora) en grupos con: ',
    '  ' + ', '.join(subgroup_df.dropna(subset=['auc_roc']).nsmallest(3, 'auc_roc')['group_value'].tolist()),
    '',
    '## Outputs',
    '',
    '- `metrics_by_subgroup.csv` — tabla completa',
    '- `top20_false_positives.csv` — usuarios con churn predicho pero no real',
    '- `top20_false_negatives.csv` — usuarios con churn real pero predicho activo',
]

with open(REPORT_DIR / 'error_analysis_summary.md', 'w') as f:
    f.write('\n'.join(lines))
print('error_analysis_summary.md guardado')

error_analysis_summary.md guardado
